# SR830 XY-only harmonic response

Read-only plotting for the paired SR830 harmonic record. XX readings are discarded by the loader. Each plotted point is annotated with its harmonic order (h1, h2, h3).

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display

from attodry_control.commissioning_analysis import browse_commissioning_file
from attodry_control.xy_harmonic_analysis import (
    discover_xy_harmonic_records,
    load_xy_harmonic_samples,
    plot_xy_harmonics,
)

working_directory = Path.cwd().resolve()
PROJECT_ROOT = (
    working_directory.parent
    if working_directory.name.lower() == 'notebooks'
    else working_directory
)
DATA_DIRECTORY = PROJECT_ROOT / 'run_data' / 'commissioning'
DATA_DIRECTORY

## Choose a completed harmonic record

The newest completed harmonic JSON is selected automatically. Set `OPEN_BROWSER=True` to choose a file through the Windows Browse dialog. Rejected records remain blocked unless `INCLUDE_REJECTED=True` is set explicitly.

In [ ]:
OPEN_BROWSER = False
INCLUDE_REJECTED = False
selected_path = None
if OPEN_BROWSER:
    selected_path = browse_commissioning_file(DATA_DIRECTORY)
harmonic_records = discover_xy_harmonic_records(
    DATA_DIRECTORY,
    record_statuses={'rejected'} if INCLUDE_REJECTED else {'completed'},
)
HARMONIC_PATH = selected_path or (harmonic_records[0] if harmonic_records else None)
if HARMONIC_PATH is None:
    raise FileNotFoundError('No completed XY harmonic record was found.')
HARMONIC_PATH

## Load XY only and mark harmonic order

In [ ]:
SAMPLE_STATUSES = {'clean'}
xy_rows = load_xy_harmonic_samples(
    HARMONIC_PATH,
    include_rejected=INCLUDE_REJECTED,
    sample_statuses=SAMPLE_STATUSES,
)
[(f'h{row.harmonic}', row.amplitude_v, row.phase_deg) for row in xy_rows]

## XY-only plots

The plotting function accepts only the XY rows returned above. Every point is labeled h1/h2/h3; no XX series is passed to the figure.

In [ ]:
for metric in ('x_v', 'y_v', 'amplitude_v', 'phase_deg'):
    figure = plot_xy_harmonics(xy_rows, metric=metric)
    display(figure)
    plt.close(figure)
